# 04 — Final Evaluation (Held-Out Test Set)

> **TEST-SET POLICY.** This notebook is the ONLY place in the paper pipeline (01-05) where test-set labels are used. Model hyperparameters (notebook 02) and calibration method (notebook 03) are already frozen before this notebook runs — it only **computes** metrics against those frozen choices, never searches over alternatives using test performance.
>
> **Every number below is computed from real test-set predictions — nothing is simulated or fabricated.**

In [ ]:
import sys
sys.path.append('..')

from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_curve, roc_curve

from src.config import load_config
from src.reproducibility import set_global_seed, get_logger, log_run_metadata
from src.preprocess import (
    load_cohort_features, drop_duplicate_rows, patient_level_split, check_patient_overlap,
    drop_identifier_columns, IDENTIFIER_COLUMNS,
)
from src.train_paper_models import build_long_format_predictions
from src.evaluate_paper_final import (
    bootstrap_ci, compute_threshold_metrics, load_calibrators, apply_all_methods,
    _check_and_record_test_usage,
)

config = load_config()
seed = config["project"]["random_seed"]
set_global_seed(seed)
logger = get_logger(log_file=config["logging"]["log_file"])

pe_config = config["paper_final_evaluation"]
_check_and_record_test_usage(pe_config["test_usage_marker_path"], logger)
logger.info("04_final_evaluation started (seed=%d) \u2014 TEST SET IN USE", seed)

ID_COL = config["preprocessing"]["id_column"]
TARGET_COL = config["preprocessing"]["target_column"]
MODELS_DIR = Path(pe_config["models_output_dir"])
TABLES_DIR = Path(pe_config["tables_output_dir"])
FIGURES_DIR = Path(pe_config["figures_output_dir"])
N_BOOTSTRAP = pe_config["n_bootstrap"]
ECE_BINS = pe_config["ece_bins"]
THRESHOLDS = sorted({pe_config["decision_threshold"], 0.5})  # requested threshold + 0.5 for reference

for d in [TABLES_DIR, FIGURES_DIR, Path(pe_config["predictions_output_dir"])]:
    d.mkdir(parents=True, exist_ok=True)

## 1. Unlock the test set, apply the frozen preprocessor

Reproduces the identical patient-level split from notebooks 01-02 and transforms `test_df` using the preprocessor fit exclusively on TRAIN in notebook 01 — never refit here.

In [ ]:
raw_df = load_cohort_features(config=config)
dedup_df = drop_duplicate_rows(raw_df)

train_df, val_df, test_df = patient_level_split(
    dedup_df, id_col=ID_COL, target_col=TARGET_COL,
    train_size=config["preprocessing"]["train_size"],
    val_size=config["preprocessing"]["val_size"],
    test_size=config["preprocessing"]["test_size"],
    seed=seed,
)
check_patient_overlap(train_df, val_df, test_df, id_col=ID_COL)
del train_df, val_df
logger.info("Test set unlocked: n=%d", len(test_df))

preprocessor = joblib.load(config["preprocessing"]["preprocessor_output_path"])
test_features_df = drop_identifier_columns(test_df, IDENTIFIER_COLUMNS)
X_test = preprocessor.transform(test_features_df)
y_test = test_features_df[TARGET_COL].reset_index(drop=True).to_numpy()

print(f"X_test: {X_test.shape}, positive rate: {y_test.mean():.1%}")

## 2. Load frozen models + calibrators, generate test predictions

In [ ]:
model_filenames = {"logistic_regression": "model_logreg.joblib", "lightgbm": "model_lgbm.joblib"}
models = {name: joblib.load(MODELS_DIR / fname) for name, fname in model_filenames.items()}
calibrators = load_calibrators(MODELS_DIR, list(models.keys()))

predictions = {}  # model_name -> {method: y_prob}
for model_name, model in models.items():
    y_prob_uncal = model.predict_proba(X_test)[:, 1]
    predictions[model_name] = apply_all_methods(model_name, y_prob_uncal, calibrators)
    logger.info("[%s] test predictions generated for methods: %s", model_name, list(predictions[model_name].keys()))

## 3. Save test predictions (long format, all model x method combinations)

In [ ]:
pred_frames = []
id_series = test_df["stay_id"] if "stay_id" in test_df.columns else test_df[ID_COL]
for model_name, methods in predictions.items():
    for method_name, y_prob in methods.items():
        pred_frames.append(pd.DataFrame({
            "stay_id": id_series.reset_index(drop=True),
            "model_name": model_name,
            "calibration_method": method_name,
            "predicted_prob": y_prob,
            "true_label": y_test,
        }))
test_predictions_df = pd.concat(pred_frames, ignore_index=True)

test_predictions_path = Path(pe_config["test_predictions_path"])
test_predictions_df.to_parquet(test_predictions_path, index=False)
logger.info("Test predictions saved to %s (%d rows)", test_predictions_path, len(test_predictions_df))
print(f"Saved {test_predictions_path}")

## 4. Table 4: main results (AUROC, AUPRC, Brier, ECE + 95% bootstrap CI for AUROC/Brier/ECE; sensitivity/specificity/PPV/NPV/F1/balanced accuracy at the primary threshold)

In [ ]:
primary_threshold = pe_config["decision_threshold"]
main_rows = []
for model_name, methods in predictions.items():
    for method_name, y_prob in methods.items():
        logger.info("[table_4] evaluating %s/%s on the test set (n_bootstrap=%d)...", model_name, method_name, N_BOOTSTRAP)
        disc_cal = bootstrap_ci(y_test, y_prob, ece_bins=ECE_BINS, n_bootstrap=N_BOOTSTRAP, seed=seed)
        thresh = compute_threshold_metrics(y_test, y_prob, primary_threshold)
        main_rows.append({
            "model": model_name, "calibration_method": method_name, "n_test": len(y_test),
            **disc_cal, **{f"{k}_at_primary_threshold" if k != "threshold" else "primary_threshold": v for k, v in thresh.items()},
        })

table4_main_results = pd.DataFrame(main_rows)
table4_path = Path(pe_config["main_results_table_path"])
table4_main_results.to_csv(table4_path, index=False)
logger.info("Table 4 (main results) saved to %s", table4_path)
table4_main_results.round(4)

## 5. Table 5: metrics at selected clinical decision thresholds

In [ ]:
threshold_rows = []
for model_name, methods in predictions.items():
    for method_name, y_prob in methods.items():
        for threshold in THRESHOLDS:
            metrics = compute_threshold_metrics(y_test, y_prob, threshold)
            threshold_rows.append({"model": model_name, "calibration_method": method_name, **metrics})

table5_clinical_thresholds = pd.DataFrame(threshold_rows)
table5_path = Path(pe_config["clinical_thresholds_table_path"])
table5_clinical_thresholds.to_csv(table5_path, index=False)
logger.info("Table 5 (clinical thresholds) saved to %s", table5_path)
table5_clinical_thresholds.round(4)

## 6. ROC curves (all models, calibrated and uncalibrated)

In [ ]:
from sklearn.metrics import roc_auc_score

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance")

for model_name, methods in predictions.items():
    for method_name, y_prob in methods.items():
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auroc = roc_auc_score(y_test, y_prob)
        linestyle = "-" if method_name != "uncalibrated" else ":"
        ax.plot(fpr, tpr, linestyle=linestyle, label=f"{model_name} / {method_name} (AUROC={auroc:.3f})")

ax.set_xlabel("False Positive Rate (1 \u2212 Specificity)")
ax.set_ylabel("True Positive Rate (Sensitivity)")
ax.set_title("ROC Curves \u2014 Held-Out Test Set (all models, calibrated + uncalibrated)")
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()

roc_base = FIGURES_DIR / "roc_curves_final"
fig.savefig(roc_base.with_suffix(".png"), dpi=300, bbox_inches="tight")
fig.savefig(roc_base.with_suffix(".pdf"), bbox_inches="tight")
plt.close(fig)
logger.info("ROC curve figure saved: %s.png / .pdf", roc_base)
print(f"Saved {roc_base}.png and .pdf")

## 7. Precision-recall curves (all models, calibrated and uncalibrated)

In [ ]:
from sklearn.metrics import average_precision_score

fig, ax = plt.subplots(figsize=(7, 7))
prevalence = y_test.mean()
ax.axhline(prevalence, linestyle="--", color="gray", label=f"Prevalence baseline ({prevalence:.3f})")

for model_name, methods in predictions.items():
    for method_name, y_prob in methods.items():
        precision, recall, _ = precision_recall_curve(y_test, y_prob)
        auprc = average_precision_score(y_test, y_prob)
        linestyle = "-" if method_name != "uncalibrated" else ":"
        ax.plot(recall, precision, linestyle=linestyle, label=f"{model_name} / {method_name} (AUPRC={auprc:.3f})")

ax.set_xlabel("Recall (Sensitivity)")
ax.set_ylabel("Precision (PPV)")
ax.set_title("Precision-Recall Curves \u2014 Held-Out Test Set (all models, calibrated + uncalibrated)")
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()

pr_base = FIGURES_DIR / "pr_curves_final"
fig.savefig(pr_base.with_suffix(".png"), dpi=300, bbox_inches="tight")
fig.savefig(pr_base.with_suffix(".pdf"), bbox_inches="tight")
plt.close(fig)
logger.info("PR curve figure saved: %s.png / .pdf", pr_base)
print(f"Saved {pr_base}.png and .pdf")

## 8. Run metadata

In [ ]:
log_run_metadata(
    output_path=pe_config["run_metadata_path"],
    seed=seed,
    extra={
        "notebook": "04_final_evaluation",
        "n_test": len(y_test),
        "n_bootstrap": N_BOOTSTRAP,
        "ci_metrics": ["auroc", "brier_score", "ece"],
        "primary_threshold": primary_threshold,
        "test_set_used": True,
    },
)
logger.info("04_final_evaluation finished")
print(f"N test = {len(y_test)}, deaths in test = {int(y_test.sum())}")